# <center> Employee Retention Prediction <center>
![Screenshot%202025-08-28%20125123.png](attachment:Screenshot%202025-08-28%20125123.png)

### Agenda:
1. [Frame the Problem / Business Case](#problem)
2. [Gathering the Data](#data)
3. [Data Pre-Pocessing](#processing)
4. [EDA](#eda)
5. [Feature Engineering](#engg)
6. [Model Training, Evaluation and Selection](#model)

<a id='problem'></a>

## 1. Frame the Problem / Business Case

### Background:
- Hiring and retaining employees are extremely complex tasks that require capital, time and skills
- Small business owners spend **40% of their working** hours on tasks that do **not** generate any income such as hiring
- Companies spend **15%-20%** of the employee's salary to recruit a new candidate
- An average company loses anywhere **between 1% and 2.5%** of their total revenue on the time it takes to bring a new hire up to speed
- It takes **52 days** on average to fill a position

### Problem Statement:
- You work as a data scientist at a Multi-National Corporation
- The HR team collected extensive data on their employees and approached you to develop a model that could predict which employees are more likely to quit
- The team provided you with an extensive dataset, a sample of which is provided:
    - JobInvolvement
    - Education
    - JobSatisfaction
    - PerformanceRating
    - RelationshipSatisfaction
    - WorkLifeBalance

<a id='data'></a>

## 2. Gathering the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from IPython.display import display

print("Libraries imported succesfully!")


In [ ]:
uploaded = files.upload()
file_name = next(iter(uploaded.keys()))
df = pd.read_csv(file_name)
print("Dataset loaded successfully!")

In [ ]:
display(df.head())

In [ ]:
display(df.tail())

In [ ]:
print("Dataset shape:", df.shape)

In [ ]:
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
display(df.describe().T)

In [ ]:
display(df.describe(include="all").T)

<a id='processing'></a>

## 3. Data Pre-Pocessing

In [ ]:
print("missing values per colum")
print(df.isnull().sum())

In [ ]:
print("\n total missing values")
print(df.isnull().sum().sum())

In [ ]:
print("\n duplicate rows")
print(df.duplicated().sum())

In [ ]:

df_original = df.copy()


columns_to_drop = [
    "EmployeeCount",
    "Over18",
    "StandardHours",
    "EmployeeNumber"
]

df = df.drop(
    columns=columns_to_drop,
    errors="ignore"
)

print("Unnecessary columns removed.")
print("Updated dataset shape:", df.shape)

display(df.head())

In [ ]:
print("Attrition counts:")
print(df["Attrition"].value_counts())

print("\nAttrition percentage:")
print(
    (df["Attrition"].value_counts(normalize=True) * 100)
    .round(2))

<a id='eda'></a>

## 4. Exploratory Data Analysis (EDA)

In [ ]:
#1. Attrition distribution
plt.figure(figsize=(6,4))

sns.countplot(data=df, x="Attrition")

plt.title("Employee Attrition distribution")
plt.xlabel("Attrition")
plt.ylabel("Number of Employees")

plt.tight_layout()
plt.show()

In [ ]:
#2. Attrition by Job satisfaction
plt.figure(figsize=(6,4))

sns.countplot(data=df, x="JobSatisfaction", hue="Attrition")

plt.title("Attrition by JobSatisfaction")
plt.xlabel("Employee job satisfaction")
plt.ylabel("Number of Employees")

plt.tight_layout()
plt.show()

In [ ]:
#3. Attrition by Work life balance
plt.figure(figsize=(6,4))

sns.countplot(data=df, x="WorkLifeBalance", hue="Attrition")

plt.title("Attrition by Work life balance")
plt.xlabel("Work life balance")
plt.ylabel("Number of Employees")

plt.tight_layout()
plt.show()

In [ ]:
#4. Attrition by Overtime
plt.figure(figsize=(6,4))

sns.countplot(data=df, x="OverTime", hue="Attrition")

plt.title("Attrition by Overtime")
plt.xlabel("Overtime")
plt.ylabel("Number of Employees")

plt.tight_layout()
plt.show()

In [ ]:
#5. Attrition by Department
plt.figure(figsize=(6,4))

sns.countplot(data=df, x="Department", hue="Attrition")

plt.title("Attrition by Department")
plt.xlabel("Department")
plt.ylabel("Number of Employees")


plt.tight_layout()
plt.show()

In [ ]:
#6. Attrition by Age
plt.figure(figsize=(6,4))

sns.histplot(data=df, x="Age", hue="Attrition", multiple="stack", bins=20)

plt.title("Attrition by Age")
plt.xlabel("Age")
plt.ylabel("Number of Employees")


plt.tight_layout()
plt.show()

In [ ]:
#7. Attrition by Monthly Income
plt.figure(figsize=(6,4))

sns.boxplot(data=df, x="Attrition", y="MonthlyIncome")
plt.title("Attrition by Monthly Income")
plt.xlabel("Attrition")
plt.ylabel("Monthly income")


plt.tight_layout()
plt.show()

In [ ]:
#8. Correlation Heatmap

numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(14, 10))

sns.heatmap(
    numeric_df.corr(),
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Heatmap of Numerical Features")
plt.tight_layout()
plt.show()


In [ ]:
# EDA Summary Statistics

print("Total employees:", len(df))

print("\nAttrition distribution:")
display(df["Attrition"].value_counts().to_frame("Count"))

print("\nAttrition rate:")
print(
    f"{(df['Attrition'].eq('Yes').mean() * 100):.2f}%"
)

print("\nAverage age by attrition:")
display(df.groupby("Attrition")["Age"].mean().round(2))

print("\nAverage monthly income by attrition:")
display(
    df.groupby("Attrition")["MonthlyIncome"]
    .mean()
    .round(2)
)

print("\nOvertime by attrition:")
display(pd.crosstab(df["OverTime"], df["Attrition"]))

<a id='engg'></a>

## 5. Feature Engineering

In [ ]:
#1. Separate input features and target


# Restore the original dataset
df = df_original.copy()

# Remove unnecessary columns
columns_to_drop = [
    "EmployeeCount",
    "Over18",
    "StandardHours",
    "EmployeeNumber"
]

df = df.drop(columns=columns_to_drop, errors="ignore")

# Clean spaces around target values
df["Attrition"] = df["Attrition"].astype("string").str.strip()

# Remove rows where the target is genuinely missing
df = df.dropna(subset=["Attrition"]).copy()

# Standardize target values
df["Attrition"] = df["Attrition"].str.lower()

# Check for unexpected values
print("Attrition values before encoding:")
print(df["Attrition"].value_counts(dropna=False))

unexpected = ~df["Attrition"].isin(["yes", "no"])

if unexpected.any():
    print("Unexpected target values:")
    print(df.loc[unexpected, "Attrition"].unique())
    raise ValueError("Please inspect the unexpected Attrition values.")

# Encode target
df["Attrition"] = df["Attrition"].map({
    "no": 0,
    "yes": 1
})

# Separate features and target
X = df.drop(columns=["Attrition"])
y = df["Attrition"]

print("\nMissing values in y:", y.isna().sum())
print("X shape:", X.shape)
print("y shape:", y.shape)



In [ ]:
#2. Split Training and Testing data

from sklearn.model_selection import train_test_split

valid_idx = y.notna()
X_clean = X[valid_idx]
y_clean = y[valid_idx]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


In [ ]:
#3. Create Preprocessing Pipelines


from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.base import clone
import numpy as np

# Identify numerical and categorical columns
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessor created successfully!")



<a id='model'></a>

## 6. Model Training, Evaluation and Selection

In [ ]:
#1. Train Four Machine learning model

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter = 2000,
        class_weight = "balanced",
        random_state = 42
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight = "balanced",
        random_state = 42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators = 200,
        class_weight = "balanced",
        random_state = 42
    ),

    "Gradient Boosting" : GradientBoostingClassifier(
        random_state = 42
    )

}

print("Models defined successfully.")

In [ ]:
#2. Train and Evaluate models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

trained_models = {}
results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", model)
    ])

    # Train the model
    pipeline.fit(X_train, y_train)

    # Predict the test set
    y_pred = pipeline.predict(X_test)

    # Predict probabilities of attrition
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Store the trained pipeline
    trained_models[name] = pipeline

    # Record evaluation metrics
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

# Display model comparison
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

display(results_df.round(3))



In [ ]:
#3. Select the best performing model

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Selected model:", best_model_name)

print("\nPerformance:")
display(
    results_df[results_df["Model"] == best_model_name]
    .round(3)
)

In [ ]:
#4. Classification report and Confusion matrix

from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay
)

y_pred_best = best_model.predict(X_test)

print(classification_report(
    y_test,
    y_pred_best,
    labels=[0, 1],
    target_names=["Stayed", "Left"],
    zero_division=0
))

ConfusionMatrixDisplay.from_estimator(
    best_model,
    X_test,
    y_test,
    display_labels=["Stayed", "Left"],
    cmap="Blues"
)

plt.title("Confusion Matrix - Selected Model")
plt.tight_layout()
plt.show()

In [ ]:
#Saving the model

import joblib
from google.colab import files

joblib.dump(
    best_model,
    "employee_attrition_model.pkl"
)

files.download("employee_attrition_model.pkl")